In [ ]:
from rag import *
from datasets import load_dataset
import json

In [ ]:
# Main evaluation pipeline
# Load dataset
dataset = load_dataset(
    "framolfese/2WikiMultihopQA",
    split="validation"
)

# Select evaluation questions
eval_dataset = dataset.shuffle(seed=42).select(range(3)) # Adjust the range for more evaluation questions

# Build full validation corpus
documents = load_2wiki_corpus(dataset)

In [ ]:
# Initialize model
rag = RAG(
    model_id="state-spaces/mamba3-siso-1.5b",
    tokenizer_id="meta-llama/Llama-3.1-8B",
    device="cuda",
    dtype=torch.bfloat16,
    space="cosine",
    dim=2048
)

In [ ]:
# Limit context
s = set()

for i in range(len(eval_dataset)):
    [s.add(title) for title in eval_dataset[i]['context']['title']]

s = list(s)

docs = {t: documents[t] for t in s}

In [ ]:
# Ingest corpus
rag.ingest(
    docs,
    strategy="baseline",
    chunk_size=512,
    overlap=0.1
)

(22, 2048)
(22,)
Documents: 22
Chunks: 22


In [ ]:
# Evaluate
results = evaluate(
    rag,
    eval_dataset,
    k=5,
    max_new_tokens=100
)

# Save results
with open("evaluation_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4, ensure_ascii=False)

Where was the director of film Breakup Buddies born?

        Answer the following question using only the provided context.

        If the context does not contain enough information,
        state that you cannot determine the answer.

        Context:
        Source: Ian Barry (director)
Content: Ian Barry is an Australian director of film and TV.

Source: J. Searle Dawley
Content: James Searle Dawley (May 13, 1877 – March 30, 1949) was an American producer, director and screenwriter. He directed 149 silent films between 1907 and 1926. He was born in Del Norte, Colorado, and died in Hollywood, California. Searley is best remembered for directing the Thomas Edison version of "Frankenstein" in 1910. He reportedly referred to himself as "the first motion picture director", which was not accurate. He worked for a while at Edison's studio, then started his own production company, being quite successful financially, although most of his films are little known today. When critics began to